# Day 9 v1 — Silver Transformation: Charging Sessions (Blob CSV → Silver Delta)

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions/`
**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/charging_sessions/` (Delta)

This is the **learning version** — one entity (charging_sessions), every step written explicitly.
Same teaching pattern as Day 8 v1 but for Blob CSV data instead of API JSON.

**Blob CSV Bronze structure:**
```
bronze-volume/realtime/charging_sessions/YYYY/MM/DD/HH/sessions_YYYYMMDD_HHMM.csv
```

**Steps:**
1. Read all Bronze CSV files for charging_sessions
2. Cast columns to correct types
3. Trim whitespace from string columns
4. Add Silver audit columns
5. Deduplicate on `session_id`
6. Write to Silver as Delta table (full overwrite)
7. Verify the output

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Constants
BRONZE_PATH = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/charging_sessions"
SILVER_PATH = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/charging_sessions"

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze path : {BRONZE_PATH}")
print(f"Silver path : {SILVER_PATH}")
print(f"Run time    : {RUN_TS}")

In [ ]:
# Cell 3 — Read all Bronze CSV files
# Bronze stores one CSV per hour per day:
#   charging_sessions/2026/06/01/06/sessions_20260601_0600.csv
#
# spark.read.csv() reads ALL files under the path recursively.
# header=True uses the first row as column names.
# inferSchema=True auto-detects types (we re-cast explicitly in the next cell).

raw_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(BRONZE_PATH)
)

print(f"Schema inferred from CSV:")
raw_df.printSchema()
print(f"Row count: {raw_df.count()}")
raw_df.show(3, truncate=True)

In [ ]:
# Cell 4 — Cast columns to correct types
# inferSchema may guess wrong (e.g. timestamps as strings).
# Explicit casting guarantees consistent Silver schema across all runs.

typed_df = raw_df.select(
    F.col("session_id").cast("string"),
    F.col("vehicle_id").cast("string"),
    F.col("charger_id").cast("string"),
    F.col("customer_id").cast("string"),
    F.col("station_id").cast("string"),
    F.col("start_time").cast("timestamp"),
    F.col("end_time").cast("timestamp"),
    F.col("energy_kwh").cast("decimal(10,4)"),
    F.col("duration_minutes").cast("integer"),
    F.col("status").cast("string"),
    F.col("cost_aud").cast("decimal(10,2)"),
    F.col("created_at").cast("timestamp"),
    F.col("updated_at").cast("timestamp"),
)

print("After type casting:")
typed_df.printSchema()

In [ ]:
# Cell 5 — Trim whitespace from all string columns
# CSV files can have trailing spaces in string fields.

string_cols = [c for c, t in typed_df.dtypes if t == "string"]
trimmed_df  = typed_df
for col in string_cols:
    trimmed_df = trimmed_df.withColumn(col, F.trim(F.col(col)))

print(f"Trimmed string columns: {string_cols}")

In [ ]:
# Cell 6 — Add Silver audit columns
# Same pattern as Day 8 — every Silver row gets lineage columns.

audited_df = (
    trimmed_df
    .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
    .withColumn("silver_load_type",   F.lit("full"))
    .withColumn("silver_pipeline",    F.lit("pl_silver_blob_charging_sessions_v1"))
)

print("After adding audit columns:")
audited_df.printSchema()

In [ ]:
# Cell 7 — Deduplicate on session_id
# Bronze may contain the same session_id across multiple hourly CSV files
# (e.g. an in-progress session updated across two hours).
# Keep the row with the most recent updated_at per session_id.

window = Window.partitionBy("session_id").orderBy(F.col("updated_at").desc())

deduped_df = (
    audited_df
    .withColumn("_row_num", F.row_number().over(window))
    .filter(F.col("_row_num") == 1)
    .drop("_row_num")
)

before = audited_df.count()
after  = deduped_df.count()
print(f"Before dedup : {before}")
print(f"After dedup  : {after}")
print(f"Duplicates removed : {before - after}")

In [ ]:
# Cell 8 — Write to Silver as Delta table (full overwrite)

(
    deduped_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)

print(f"Written to: {SILVER_PATH}")
print(f"Rows written: {deduped_df.count()}")

In [ ]:
# Cell 9 — Verify Silver output

silver_df = spark.read.format("delta").load(SILVER_PATH)

print("Silver charging_sessions schema:")
silver_df.printSchema()
print(f"\nTotal rows: {silver_df.count()}")
silver_df.show(5, truncate=True)

print("\nNull check on session_id (should be 0):")
print(silver_df.filter(F.col("session_id").isNull()).count())

print("\nStatus distribution:")
silver_df.groupBy("status").count().orderBy("count", ascending=False).show()